# AKANE E2-2：つながった地理テストで、自転車・車・ロケットを比べる

**問い**：問題どうしがつながったテストで、正解をルールとは別のところ（世界地名データ GeoNames）から取ったとき、
網全体で考える合図（**BP＝車**、**p-bitの揺れ＝ロケット**）は、**数える合図（自転車）**や**口の自信**より、口（Gemma-2 2B）の間違いをよく拾うか。

**テスト**：「〇〇市は△△にある」「△△は□□の国」「〇〇市は□□にある」という○×問題を、本物と偽物を混ぜて約3,600問。
**ルール**：「市が国にあり、国が大陸にあるなら、市はその大陸にある」「1つの市は1つの国にしかない」「1つの市・国は1つの大陸にしかない」。
国や大陸を通して問題がつながるので、2段・3段先の食い違いが出る。

**合図**：1 口の自信　2 数える（破れたルールの数）　3 MaxSATが変えたか　4 BP（口と反対になる確率）　5 p-bitの揺れ（口と反対になった割合）

**先に固定した基準**（sims/e2_geo/PLAN.md）
- 「AがBに勝つ」＝10回平均のAUROC差が0.02以上、かつ10回中8回以上でAが上。差0.02未満は「同着」。
- 主に見るのは全体での「BP対数える」「揺れ対数える」。どちらも勝たなければ「つながったテストでも、網全体で考える意味は見えなかった」と記録する。
- 結果を見てから設定を変えない。

**時間の目安**（T4 GPU）：全部で15〜25分。**パソコンからの実行がおすすめ**（スマホだと途中で切れやすい）。

## 0. 準備
1. 「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU**。
2. 鍵マークの `HF_TOKEN`（E1で入れたもの）にアクセスを許可。
3. 「ランタイム」→「すべてのセルを実行」。

In [ ]:
# ===== 設定 =====
USE_MOCK_MOUTH = False      # True: 作り物の口（動作確認用。結果は研究に使わない）
SAVE_TO_DRIVE  = True       # 結果をドライブの AKANE_アーキテクチャ/E2_results に保存
MODEL_ID = "google/gemma-2-2b-it"
SPLITS = [0, 1, 2, 3, 4]    # 調整用（市の20%）／評価用の分け方（5通り）
PBIT_SEEDS = [1, 2]         # p-bitの乱数（分け方ごとに2通り → 合計10回）
DATA_SEED = 0               # テストの作り方（固定）

In [ ]:
import os, json, time, itertools, collections
import numpy as np
for pkg, mod in [("python-sat", "pysat"), ("geonamescache", "geonamescache")]:
    try: __import__(mod)
    except ImportError: os.system(f"pip -q install {pkg}")
import geonamescache
from pysat.formula import WCNF
from pysat.examples.rc2 import RC2
from sklearn.metrics import roc_auc_score

OUT = "/content/E2_results" if os.path.exists("/content") else "E2_results"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/AKANE_アーキテクチャ/E2_results"
os.makedirs(OUT, exist_ok=True)
print("保存先:", OUT)

## 1. テストを作る（正解は GeoNames。ルールとは別のところから来る）
- 国は、大陸をまたぐ国（ロシア、トルコ、エジプトなど）を除く。市は、同じ名前が他にない市だけ。
- 1つの国から4市（一番大きい市＋ランダムに3市）。
- 偽物：市には同じ大陸の別の国を2つ、市と国には別の大陸を1つ。

In [ ]:
gc = geonamescache.GeonamesCache()
CONT = {"AF": "Africa", "AS": "Asia", "EU": "Europe", "NA": "North America", "OC": "Oceania", "SA": "South America"}
SKIP = {"RU", "TR", "EG", "KZ", "AZ", "GE", "AM", "CY", "ID", "PG", "TL"}
cos = {k: v for k, v in gc.get_countries().items() if v["continentcode"] in CONT and k not in SKIP}
allc = list(gc.get_cities().values())
name_cnt = collections.Counter(c["name"] for c in allc)
cnames = {v["name"] for v in cos.values()}
by_co = collections.defaultdict(list)
for c in allc:
    if c["countrycode"] in cos and name_cnt[c["name"]] == 1 and c["name"] not in cnames and c["name"].isascii():
        by_co[c["countrycode"]].append(c)
COUNTRIES = [k for k in sorted(by_co) if len(by_co[k]) >= 4]
rng = np.random.default_rng(DATA_SEED)
STMT, GOLD, KIND, CITY = [], [], [], []
idx = {}
def add(key, text, gold, kind, city=None):
    if key in idx: return idx[key]
    idx[key] = len(STMT); STMT.append(text); GOLD.append(gold); KIND.append(kind); CITY.append(city)
    return idx[key]
cont_of = {k: cos[k]["continentcode"] for k in COUNTRIES}
same_cont = collections.defaultdict(list)
for k in COUNTRIES: same_cont[cont_of[k]].append(k)
others = lambda z: [w for w in CONT if w != z]
# 国→大陸
for k in COUNTRIES:
    z = cont_of[k]; fz = rng.choice(others(z))
    for zz in (z, fz):
        add(("CK", k, zz), f"{cos[k]['name']} is a country in {CONT[zz]}.", int(zz == z), "国→大陸")
# 市→国、市→大陸
CITIES = []
for k in COUNTRIES:
    cs = sorted(by_co[k], key=lambda c: -c["population"])
    pick = [cs[0]] + list(rng.choice(cs[1:], 3, replace=False))
    for c in pick:
        name = c["name"]; z = cont_of[k]; CITIES.append(name)
        fks = [w for w in same_cont[z] if w != k]
        fks = list(rng.choice(fks, min(2, len(fks)), replace=False))
        for kk in [k] + fks:
            add(("CC", name, kk), f"{name} is a city in {cos[kk]['name']}.", int(kk == k), "市→国", name)
        for zz in (z, rng.choice(others(z))):
            add(("CL", name, zz), f"{name} is located in {CONT[zz]}.", int(zz == z), "市→大陸", name)
GOLD = np.array(GOLD)
DUMMY = len(STMT)   # 「両立しない」ルールの結論に使う、いつも0の素子
# ルール：(前提1, 前提2, 結論)。結論がDUMMYなら「前提が両方1ではいけない」
RULES = set()
for (t, a, b), i in list(idx.items()):
    if t == "CC":
        for zz in CONT:
            if ("CK", b, zz) in idx and ("CL", a, zz) in idx:
                RULES.add((i, idx[("CK", b, zz)], idx[("CL", a, zz)]))     # 市が国にあり、国が大陸にある → 市はその大陸
def excl(keys):
    ids = [idx[k] for k in keys if k in idx]
    for p, q in itertools.combinations(sorted(ids), 2): RULES.add((p, q, DUMMY))
for name in set(CITIES):
    excl([("CC", name, k) for k in COUNTRIES]); excl([("CL", name, z) for z in CONT])
for k in COUNTRIES:
    excl([("CK", k, z) for z in CONT])
RULES = sorted(RULES)
# ルールが正解で全部守られているか（守られていなければデータの誤り）
G = np.append(GOLD, 0)
bad = sum(1 for p, q, h in RULES if G[p] == 1 and G[q] == 1 and G[h] == 0)
print(f"国 {len(COUNTRIES)}、市 {len(set(CITIES))}、問題 {len(STMT)}（本物 {GOLD.sum()}）、ルール {len(RULES)}本、正解が破るルール {bad}")
print("例:", STMT[0], "|", STMT[2], "|", STMT[5])

## 2. 口に聞く（1回、揺らぎなし。保存するので2回目からは飛ばせる）

In [ ]:
def mock_mouth(y, rng):
    lg = 2.0 * (2 * y - 1) + rng.normal(0, 1.5, len(y))
    flip = rng.random(len(y)) < 0.12
    lg[flip] = -np.sign(lg[flip]) * np.abs(rng.normal(2, 0.7, flip.sum()))
    return lg

cache = f"{OUT}/geo_mouth_{'MOCK' if USE_MOCK_MOUTH else MODEL_ID.split('/')[-1]}.json"
if os.path.exists(cache):
    LOGIT = np.array(json.load(open(cache))); print("保存済みの答えを読み込んだ:", cache)
elif USE_MOCK_MOUTH:
    LOGIT = mock_mouth(GOLD, np.random.default_rng(123)); json.dump(LOGIT.tolist(), open(cache, "w"))
else:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from google.colab import userdata
    tok = AutoTokenizer.from_pretrained(MODEL_ID, token=userdata.get("HF_TOKEN"), padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, token=userdata.get("HF_TOKEN"),
                                                 torch_dtype=torch.float16, device_map="cuda").eval()
    YES = tok.encode("Yes", add_special_tokens=False)[0]; NO = tok.encode("No", add_special_tokens=False)[0]
    def prompt(s):
        msg = [{"role": "user", "content": f"Is the following statement true? Answer Yes or No only.\nStatement: {s}"}]
        return tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    LOGIT = np.zeros(len(STMT)); B = 64; t = time.time()
    for i in range(0, len(STMT), B):
        enc = tok([prompt(s) for s in STMT[i:i + B]], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            last = model(**enc).logits[:, -1, :].float()
        LOGIT[i:i + B] = (last[:, YES] - last[:, NO]).cpu().numpy()
        if (i // B) % 20 == 0: print(f"{i + B}/{len(STMT)}  {time.time() - t:.0f}秒")
    json.dump(LOGIT.tolist(), open(cache, "w")); print("保存:", cache)
for k in ["市→国", "市→大陸", "国→大陸"]:
    s = np.array([x == k for x in KIND])
    print(f"口だけの正解率 {k}: {((LOGIT[s] > 0) == GOLD[s]).mean():.3f}")

## 3. 網の計算（5つの合図）
エネルギー E(x) = −λ Σ l_i x_i + W Σ_ルール［前提が両方1かつ結論が0］。ルールは全部3つ組（「両立しない」は、いつも0の素子を結論にする）。

In [ ]:
def pbit(lg, rules, lam, W, rng, chains=10, burn=100, sweeps=400, col=None):
    R = np.array(rules); n = len(lg); a, b, h = R[:, 0], R[:, 1], R[:, 2]
    col = coloring(n, rules) if col is None else col
    groups = [np.nonzero(col == c)[0] for c in range(col.max() + 1)]
    x = (rng.random((chains, n)) < 0.5).astype(float); x[:, DUMMY] = 0; acc = np.zeros(n)
    for t in range(burn + sweeps):
        for g in groups:
            dE = np.tile(-lam * lg, (chains, 1))
            xa, xb, xh = x[:, a], x[:, b], x[:, h]
            np.add.at(dE.T, a, (W * xb * (1 - xh)).T)     # a を1にすると違反になる
            np.add.at(dE.T, b, (W * xa * (1 - xh)).T)
            np.add.at(dE.T, h, (-W * xa * xb).T)          # h を0にすると違反になる
            p1 = 1 / (1 + np.exp(np.clip(dE[:, g], -50, 50)))
            x[:, g] = (rng.random((chains, len(g))) < p1).astype(float)
            x[:, DUMMY] = 0
        if t >= burn: acc += x.mean(0)
    return acc / sweeps

def coloring(n, rules):
    nb = [set() for _ in range(n)]
    for r in rules:
        for u in r:
            if u != DUMMY: nb[u].update(v for v in r if v != u and v != DUMMY)
    col = -np.ones(n, int)
    for i in np.argsort([-len(s) for s in nb]):
        used = {col[j] for j in nb[i]}; c = 0
        while c in used: c += 1
        col[i] = c
    col[DUMMY] = col.max() + 1
    return col

A8 = ((np.arange(8)[:, None] >> np.arange(3)) & 1)        # 3つ組の全部の組み合わせ
VIOL = (A8[:, 0] == 1) & (A8[:, 1] == 1) & (A8[:, 2] == 0)
def bp(lg, rules, lam, W, damp=0.5, iters=200, tol=1e-6):
    R = np.array(rules); n = len(lg); F = len(R)
    un = np.stack([np.zeros(n), lam * lg], 1)
    lt = -W * VIOL                                          # (8,)
    msg = np.zeros((F, 3, 2)); conv = False
    for it in range(iters):
        tot = un.copy()
        for j in range(3): np.add.at(tot, R[:, j], msg[:, j])
        vin = tot[R] - msg                                  # (F,3,2) 主張→ルール
        vin[R == DUMMY] = (0.0, -1e9)                       # いつも0の印は0に固定（多くのルールから押されても動かさない）
        s = lt[None, :] + sum(vin[:, j, A8[:, j]] for j in range(3))   # (F,8)
        new = np.zeros_like(msg)
        for j in range(3):
            for v in (0, 1):
                sel = A8[:, j] == v
                new[:, j, v] = np.logaddexp.reduce(s[:, sel] - vin[:, j, v][:, None], axis=1)
        new -= new.max(2, keepdims=True)
        new = damp * msg + (1 - damp) * new
        d = np.abs(new - msg).max(); msg = new
        if d < tol: conv = True; break
    tot = un.copy()
    for j in range(3): np.add.at(tot, R[:, j], msg[:, j])
    return 1 / (1 + np.exp(np.clip(tot[:, 0] - tot[:, 1], -50, 50))), conv

def maxsat(lg, rules, lam, W):
    n = DUMMY; wc = WCNF()
    for v in range(n):
        wc.append([(v + 1) if lg[v] > 0 else -(v + 1)], weight=max(1, int(round(100 * lam * abs(lg[v])))))
    for p, q, h in rules:
        wc.append([-(p + 1), -(q + 1)] + ([h + 1] if h != DUMMY else []), weight=max(1, int(round(100 * W))))
    with RC2(wc) as r: m = r.compute()
    return np.append((np.array(m[:n]) > 0).astype(int), 0)

def violations(ans, rules):
    c = np.zeros(len(ans))
    for p, q, h in rules:
        if ans[p] == 1 and ans[q] == 1 and ans[h] == 0:
            c[p] += 1; c[q] += 1; c[h] += 1
    return c

## 4. 道具の確かめ（小さな網で正確な答えと比べる。p-bitの平均誤差0.02未満で合格）

In [ ]:
def exact(lg, rules, lam, W, nodes):
    k = len(nodes); loc = {v: i for i, v in enumerate(nodes)}
    A = ((np.arange(2 ** k)[:, None] >> np.arange(k)) & 1)
    E = -lam * A @ lg[nodes]
    for p, q, h in rules:
        xh = A[:, loc[h]] if h != DUMMY else 0
        E = E + W * ((A[:, loc[p]] == 1) & (A[:, loc[q]] == 1) & (xh == 0))
    w = np.exp(-(E - E.min())); return (w @ A) / w.sum()

rng = np.random.default_rng(0); ep, eb = [], []
for name in sorted(set(CITIES))[:15]:
    k = [kk for kk in COUNTRIES if ("CC", name, kk) in idx]
    nodes = sorted({idx[("CC", name, kk)] for kk in k} | {idx[("CL", name, z)] for z in CONT if ("CL", name, z) in idx}
                   | {idx[("CK", kk, z)] for kk in k for z in CONT if ("CK", kk, z) in idx})
    S = set(nodes) | {DUMMY}
    rs = [r for r in RULES if set(r) <= S]
    lgs = np.append(rng.normal(0, 2, DUMMY), -60.0)
    ex = exact(lgs, rs, 1, 3, nodes)
    pp = pbit(lgs, rs, 1, 3, rng, chains=20, burn=200, sweeps=2000)[nodes]
    pb = bp(lgs, rs, 1, 3)[0][nodes]
    ep.append(np.abs(pp - ex).mean()); eb.append(np.abs(pb - ex).mean())
print(f"小さな網15個（各{len(nodes)}問ほど）：p-bitの誤差 {np.mean(ep):.4f}、BPの誤差 {np.mean(eb):.4f} →",
      "合格" if np.mean(ep) < 0.02 else "不合格（ここで止める）")
# 網全体でも、BPと長く回したp-bitが近いか（正確な答えは計算できないので、長いp-bitを物差しにする）
lgs = np.append(rng.normal(0, 2, DUMMY), -60.0)
pl = pbit(lgs, RULES, 1, 1, np.random.default_rng(1), chains=20, burn=500, sweeps=3000)
print(f"網全体（{DUMMY}問）：BPと長いp-bitの差 {np.abs(bp(lgs, RULES, 1, 1)[0] - pl)[:DUMMY].mean():.4f}、"
      f"いつものp-bitと長いp-bitの差 {np.abs(pbit(lgs, RULES, 1, 1, rng) - pl)[:DUMMY].mean():.4f}")

## 5. 本番（市の20%を調整用、残りを評価用。分け方5通り × p-bitの乱数2通り）
網の計算は全部の問題で行う（網は口の答えしか使わない）。正解を使うのは、調整用の目盛り合わせと評価だけ。国→大陸の問題は評価用に入れる。

In [ ]:
COL = coloring(DUMMY + 1, RULES)
city_arr = np.array([c if c else "" for c in CITY])
ucities = sorted(set(CITIES))
def nhop_info():
    # 各問題が、どれだけ多くのルールに入っているか（つながりの多さ）
    deg = np.zeros(DUMMY + 1)
    for r in RULES:
        for u in r: deg[u] += 1
    return deg
DEG = nhop_info()

def run_split(split):
    rng = np.random.default_rng(split)
    calc = set(rng.choice(ucities, int(round(0.2 * len(ucities))), replace=False))
    cal = np.array([i for i in range(DUMMY) if CITY[i] in calc])
    test = np.array([i for i in range(DUMMY) if CITY[i] not in calc])
    cl = LOGIT[cal]; scale = 2.0 / cl.std()
    shift = max(np.quantile(cl, np.linspace(0.02, 0.98, 97)), key=lambda b: ((cl > b) == GOLD[cal]).mean())
    lg = np.append((LOGIT - shift) * scale, -60.0)
    m = (lg > 0).astype(int); gold = np.append(GOLD, 0)
    best = {}
    for lam, W in itertools.product([0.5, 1, 2], [1, 2, 4]):
        acc = {"maxsat": (maxsat(lg, RULES, lam, W)[cal] == gold[cal]).mean(),
               "bp": ((bp(lg, RULES, lam, W)[0][cal] > .5) == gold[cal]).mean(),
               "pbit": ((pbit(lg, RULES, lam, W, np.random.default_rng(0), col=COL)[cal] > .5) == gold[cal]).mean()}
        for k, a in acc.items():
            if k not in best or a > best[k][0]: best[k] = (a, lam, W)
    ms = maxsat(lg, RULES, *best["maxsat"][1:])
    pb, conv = bp(lg, RULES, *best["bp"][1:])
    vio = violations(m, RULES)
    conf = np.abs(1 / (1 + np.exp(-lg)) - .5)
    err = (m != gold).astype(int)
    out = []
    for ps in PBIT_SEEDS:
        pp = pbit(lg, RULES, *best["pbit"][1:], np.random.default_rng(1000 * split + ps), col=COL)
        sig = {"口の自信": -conf, "数える": vio - conf / 10, "MaxSAT": (ms != m) - conf / 10,
               "BP": np.where(m == 1, 1 - pb, pb), "揺れ": np.where(m == 1, 1 - pp, pp),
               "揺れ_0.5への近さ（E1の測り方）": -np.abs(pp - .5)}
        subsets = {"全体": test,
                   "市→国": np.array([i for i in test if KIND[i] == "市→国"]),
                   "どのルールも破っていない主張": test[vio[test] == 0]}
        auc = {}
        for sn, S in subsets.items():
            y = err[S]
            auc[sn] = {k: (float(roc_auc_score(y, v[S])) if 0 < y.sum() < len(y) else float("nan")) for k, v in sig.items()}
            auc[sn]["主張"] = int(len(S)); auc[sn]["口の間違い"] = int(y.sum())
        acc = {"口": float((m[test] == gold[test]).mean()), "MaxSAT": float((ms[test] == gold[test]).mean()),
               "BP": float(((pb[test] > .5) == gold[test]).mean()), "揺れ": float(((pp[test] > .5) == gold[test]).mean())}
        out.append({"split": split, "pbit_seed": ps, "auc": auc, "acc": acc, "best": {k: v[1:] for k, v in best.items()},
                    "bp_converged": bool(conv)})
        if split == SPLITS[0] and ps == PBIT_SEEDS[0]:
            KEEP.update(sig=sig, test=test, err=err)
        print(f"分け方{split} 乱数{ps}: " + "  ".join(f"{k} {auc['全体'][k]:.3f}" for k in ["口の自信", "数える", "MaxSAT", "BP", "揺れ"]), flush=True)
    return out

t = time.time(); RES = []; KEEP = {}
for s in SPLITS:
    RES += run_split(s)
print(f"終わり（{time.time() - t:.0f}秒）")

## 6. 先に決めた基準で判定する

In [ ]:
SUBNAMES = ["全体", "市→国", "どのルールも破っていない主張"]
def judge(a, b, sub="全体"):
    A = np.array([r["auc"][sub][a] for r in RES]); B = np.array([r["auc"][sub][b] for r in RES])
    ok = ~(np.isnan(A) | np.isnan(B)); A, B = A[ok], B[ok]
    d = (A - B).mean(); w = int((A > B).sum()); n = len(A)
    if d >= 0.02 and w >= 0.8 * n: v = f"{a}の勝ち"
    elif d <= -0.02 and (B > A).sum() >= 0.8 * n: v = f"{b}の勝ち"
    elif abs(d) < 0.02: v = "同着"
    else: v = "はっきりしない"
    return f"{a} 対 {b}：差 {d:+.3f}（{w}/{n}回で{a}が上）→ {v}"

print("口:", "作り物（動作確認。研究結果ではない）" if USE_MOCK_MOUTH else MODEL_ID)
for sub in SUBNAMES:
    r0 = RES[0]["auc"][sub]
    print(f"\n【{sub}】主張 {r0['主張']}、口の間違い {r0['口の間違い']}（分け方0）")
    for k in ["口の自信", "数える", "MaxSAT", "BP", "揺れ", "揺れ_0.5への近さ（E1の測り方）"]:
        print(f"  {k:22s} AUROC {np.nanmean([r['auc'][sub][k] for r in RES]):.3f}")
    for a, b in [("BP", "数える"), ("揺れ", "数える"), ("BP", "口の自信"), ("揺れ", "口の自信"), ("揺れ", "BP")]:
        print("  " + judge(a, b, sub))
print("\n直した後の正解率:", {k: round(np.mean([r["acc"][k] for r in RES]), 3) for k in ["口", "MaxSAT", "BP", "揺れ"]})
print("BPが収束した回数:", sum(r["bp_converged"] for r in RES), "/", len(RES))
fn = f"{OUT}/E2-2_geo_{'MOCK' if USE_MOCK_MOUTH else MODEL_ID.split('/')[-1]}_{time.strftime('%Y%m%d_%H%M')}.json"
json.dump(RES, open(fn, "w"), ensure_ascii=False, indent=1); print("保存:", fn)

## 7. 図で見る（note用。図は E2_results にも保存される）
左：間違いを拾う力（AUROC）。棒は10回の平均、細い線は最小〜最大。右：怪しいものから外したとき、残った答えの正解率（分け方0）。

In [ ]:
import subprocess, matplotlib, matplotlib.pyplot as plt
from matplotlib import font_manager
if not any("Noto Sans CJK" in f.name for f in font_manager.fontManager.ttflist):
    subprocess.run("apt-get -qq install -y fonts-noto-cjk > /dev/null", shell=True)
    for p in font_manager.findSystemFonts():
        if "NotoSansCJK" in p: font_manager.fontManager.addfont(p)
JP = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK" in f.name]
plt.rcParams["font.family"] = JP[0] if JP else "DejaVu Sans"
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
SIGS = [("口の自信", "口の自信", "#b5b3ad"), ("数える", "数える（自転車）", "#2a78d6"), ("MaxSAT", "MaxSAT", "#eda100"),
        ("BP", "BP（車）", "#eb6834"), ("揺れ", "p-bitの揺れ（ロケット）", "#1baf7a")]
SUBS = [("全体", "全体"), ("市→国", "「市は〇〇国にある」\nの問題"), ("どのルールも破っていない主張", "どのルールも\n破っていない主張")]
fig, ax = plt.subplots(1, 2, figsize=(15, 5.4), gridspec_kw={"width_ratios": [1.25, 1]})
bw = 0.15; lo_all = 1
for gi, (sk, sl) in enumerate(SUBS):
    for si, (k, lab, c) in enumerate(SIGS):
        v = np.array([r["auc"][sk][k] for r in RES], float); v = v[~np.isnan(v)]
        if not len(v): continue
        x = gi + (si - 2) * (bw + 0.02); lo_all = min(lo_all, v.min())
        ax[0].bar(x, v.mean() - 0.5, bw, bottom=0.5, color=c, edgecolor="white", linewidth=1.5, label=lab if gi == 0 else None)
        ax[0].plot([x, x], [v.min(), v.max()], color=INK, lw=1)
        if k in ("口の自信", "数える", "揺れ"):
            ax[0].text(x, v.max() + 0.006, f"{v.mean():.2f}", ha="center", va="bottom", fontsize=9, color=INK)
ax[0].set_xticks(range(len(SUBS))); ax[0].set_xticklabels([s[1] for s in SUBS], fontsize=10, color=INK)
ax[0].set_ylim(0.5, 1.0); ax[0].set_ylabel("間違いを拾う力（AUROC）", color=INK)
ax[0].set_title("どの合図が、口の間違いをよく拾うか", fontsize=12, loc="left", color=INK)
ax[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=5, frameon=False, fontsize=9, handlelength=1.2)
sig, test, err = KEEP["sig"], KEEP["test"], KEEP["err"]
fr = np.linspace(0, 0.5, 26); ends = {}
for k, lab, c in SIGS:
    if k == "MaxSAT": continue
    s = sig[k][test] + 1e-9 * np.random.default_rng(0).random(len(test))
    ok = (err[test] == 0)[np.argsort(s)]
    y = [ok[: max(1, int(round(len(ok) * (1 - f))))].mean() * 100 for f in fr]
    ax[1].plot(fr * 100, y, color=c, lw=2); ends[k] = (y[-1], lab, c)
lo, hi = ax[1].get_ylim(); span = hi - lo; placed = []
for k, (yv, lab, c) in sorted(ends.items(), key=lambda kv: kv[1][0]):
    yy = yv
    for p in placed:
        if abs(yy - p) < 0.05 * span: yy = p + 0.05 * span
    placed.append(yy)
    ax[1].text(51, yy, lab, color=INK, fontsize=9, va="center")
ax[1].set_xlim(0, 68); ax[1].set_xlabel("怪しいものから外した割合（%）", color=INK); ax[1].set_ylabel("残った答えの正解率（%）", color=INK)
ax[1].set_title("怪しいものから外すと、残りはどれだけ正しくなるか", fontsize=12, loc="left", color=INK)
for a in ax:
    a.grid(axis="y", color=GRID, lw=0.8); a.set_axisbelow(True)
    for s in ["top", "right"]: a.spines[s].set_visible(False)
    for s in ["left", "bottom"]: a.spines[s].set_color(GRID)
    a.tick_params(colors=MUTED)
tag = "作り物の口（動作確認。研究結果ではない）" if USE_MOCK_MOUTH else MODEL_ID
fig.suptitle(f"E2-2　つながった地理テスト（正解はGeoNames）　口：{tag}　分け方5通り×p-bit乱数2通り", fontsize=13, x=0.01, ha="left", color=INK)
fig.tight_layout()
png = f"{OUT}/E2-2_図_{'MOCK' if USE_MOCK_MOUTH else MODEL_ID.split('/')[-1]}_{time.strftime('%Y%m%d_%H%M')}.png"
fig.savefig(png, dpi=150, bbox_inches="tight", facecolor="white"); plt.show(); print("図を保存:", png)

## この実験で言えないこと
- 分野は地理だけ。口は1種類。英語の文で聞いている。
- ルールは人が書いた正しいルール（本物の運用ではルールも機械が作る）。
- 偽物の問題は「同じ大陸の別の国」「別の大陸」から機械的に作った。
- p-bitはシミュレーション。速さや電力は言えない。